# Test that CUDA is available

In [1]:
!nvidia-smi

Mon Sep 21 20:18:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Constants
import os

REPO_URL = "https://github.com/gam-354/cuda_playground"
REPO_NAME = "cuda_playground"

EXERCISES_DIR = os.path.join(REPO_NAME, "exercises")
COMMON_INC = os.path.join(REPO_NAME, "common")
BIN_NAME = "out_app"

# Examples from CUDA by Example book

### PULL changes

In [3]:
# Restore progress from last time

import os

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    !cd {REPO_NAME} && git pull

Cloning into 'cuda_playground'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 54 (delta 21), reused 51 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 92.64 KiB | 13.23 MiB/s, done.
Resolving deltas: 100% (21/21), done.


### Check changes

In [4]:
!cd {REPO_NAME} && git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


# Quick exercise runner

In [13]:
def build_and_run(exercise_id):
    file = next((f for f in os.listdir(EXERCISES_DIR) if f.startswith(exercise_id) and f.endswith(".cu")), None)
    full_path = os.path.join(EXERCISES_DIR, file)

    # Make sure the file exists
    if not os.path.exists(full_path):
        print(f"Error: File {full_path} does not exist.")
        return

    print("Selected file: " + full_path)

    # Remove the previous executable
    !rm {BIN_NAME}

    # Build including common/
    !nvcc -I{COMMON_INC} {full_path} -o {BIN_NAME}

    # Run!
    !./{BIN_NAME}

In [6]:
%%time

# Last minute sync
!cd {REPO_NAME} && git pull -q

# ID of the exercise
id = "0401"

# Build and run the selected exercise
build_and_run(id)

Selected file: cuda_playground/exercises/0401_add_vectors.cu
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
0 + 0 = 0
-1 + 1 = 0
-2 + 4 = 2
-3 + 9 = 6
-4 + 16 = 12
-5 + 25 = 20
-6 + 36 = 30
-7 + 49 = 42
-8 + 64 = 56
-9 + 81 = 72
-10 + 100 = 90
-11 + 121 = 110
-12 + 144 = 132
-13 + 169 = 156
-14 + 196 = 182
-15 + 225 = 210
-16 + 256 = 240
-17 + 289 = 272
-18 + 324 = 306
-19 + 361 = 342
-20 + 400 = 380
-21 + 441 = 420
-22 + 484 = 462
-23 + 529 = 506
-24 + 576 = 552
-25 + 625 = 600
-26 + 676 = 650
-27 + 729 = 702
-28 + 784 = 756
-29 + 841 = 812
-30 + 900 = 870
-31 + 961 = 930
-32 + 1024 = 992
-33 + 1089 = 1056
-34 + 1156 = 1122
-35 + 1225 = 1190
-36 + 1296 = 1260
-37 + 1369 = 1332
-38 + 1444 = 1406
-39 + 1521 = 1482
-40 + 1600 = 1560
-41 + 1681 = 1640
-42 + 1764 = 1722
-43 + 1849 = 1806
-44 + 1936 = 1892
-45 + 2025 = 1980
-46 + 2116 = 2070
-47 + 2209 

### Quick Exercise editor

Once finished, copy it to your local repo and commit from your PC

In [17]:
%%writefile {EXERCISES_DIR}/0402_julia.cu

#include "../common/book.h"
#include "../common/cpu_bitmap.h"

#define DIM 1000

///////////////////////////////////// WARNING ///////////////////
/// THIS EXERCISE IS NOT ABLE TO LINK IN A GPU COLAB SERVER /////

// Complex number struct
struct cuComplex {
    float r;
    float i;
    
    __device__ cuComplex( float a, float b ) : r(a), i(b) {}
    
    __device__ float magnitude2( void ) {
        return r * r + i * i;
    }
    
    __device__ cuComplex operator*(const cuComplex& a) {
        return cuComplex(r*a.r - i*a.i, i*a.r + r*a.i);
    }
    
    __device__ cuComplex operator+(const cuComplex& a) {
        return cuComplex(r+a.r, i+a.i);
    }
};

// Julia function
__device__ int julia( int x, int y ) {
    const float scale = 1.5;
    float jx = scale * (float)(DIM/2 - x)/(DIM/2);
    float jy = scale * (float)(DIM/2 - y)/(DIM/2);
    cuComplex c(-0.8, 0.156);
    cuComplex a(jx, jy);
    int i = 0;
    for (i=0; i<200; i++) {
        a = a * a + c;
        if (a.magnitude2() > 1000)
            return 0;
    }
    return 1;
}

// Kernel
__global__ void kernel( unsigned char *ptr ) {
    // map from threadIdx/BlockIdx to pixel position
    int x = blockIdx.x;
    int y = blockIdx.y;
    int offset = x + y * gridDim.x;

    // now calculate the value at that position
    int juliaValue = julia( x, y );
    ptr[offset*4 + 0] = 255 * juliaValue;
    ptr[offset*4 + 1] = 0;
    ptr[offset*4 + 2] = 0;
    ptr[offset*4 + 3] = 255;
}

// Main
int main( void ) {
    CPUBitmap bitmap( DIM, DIM );
    unsigned char *dev_bitmap;
    HANDLE_ERROR( cudaMalloc( (void**)&dev_bitmap, bitmap.image_size() ) );
    dim3 grid(DIM,DIM);
    kernel<<<grid,1>>>( dev_bitmap );
    HANDLE_ERROR( cudaMemcpy( bitmap.get_ptr(), dev_bitmap, bitmap.image_size(), cudaMemcpyDeviceToHost ) );
    bitmap.display_and_exit();
    HANDLE_ERROR( cudaFree( dev_bitmap ) );
}

Overwriting cuda_playground/exercises/0402_julia.cu


In [16]:
build_and_run("0402")

Selected file: cuda_playground/exercises/0402_julia.cu
rm: cannot remove 'out_app': No such file or directory
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
In file included from cuda_playground/exercises/../common/gl_helper.h:44,
                 from cuda_playground/exercises/../common/cpu_bitmap.h:20,
                 from cuda_playground/exercises/0402_julia.cu:3:
cuda_playground/common/GL/glut.h:151: warning: "APIENTRY" redefined
  151 | # define APIENTRY
      | 
In file included from cuda_playground/common/GL/glut.h:137:
/usr/include/GL/gl.h:83: note: this is the location of the previous definition
   83 | #define APIENTRY GLAPIENTRY
      | 
cuda_playground/common/GL/glut.h(158): warning #541-D: allowing all exceptions is incompatible with previous function "exit" (declared at line 756 of /usr/include/stdlib.h)
        void exit(int);
     